# config

> Shapes for the Audio-JEPA context encoder.
>
> The model is a ViT-Base on a 10 s Kaldi log-mel spectrogram. A 16 × 16
> patch embed turns 256 time bins × 128 mels into 128 tokens. These
> defaults match `ltuncay/Audio-JEPA`.


In [ ]:
#| default_exp config


In [ ]:
#| hide
from nbdev.showdoc import *


The checkpoint is the AudioSet-pretrained context encoder. Audio is 32 kHz mono. The hop is `10 s / 256` ≈ 39.06 ms, and each patch covers 16 frames, so one time step of the token grid is about 625 ms.


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
from dataclasses import dataclass


In [ ]:
#| export
@dataclass(frozen=True)
class AudioJEPAConfig:
    """Audio-JEPA context encoder. Frontend fields describe the Kaldi fbank."""

    sample_rate: int = 32_000
    clip_length: float = 10.0
    n_mels: int = 128
    time_bins: int = 256
    patch_h: int = 16
    patch_w: int = 16
    in_chans: int = 1
    embed_dim: int = 768
    depth: int = 12
    num_heads: int = 12
    mlp_ratio: float = 4.0
    qkv_bias: bool = True
    layer_norm_eps: float = 1e-5
    f_min: float = 20.0

    def __post_init__(self) -> None:
        if self.time_bins % self.patch_h:
            raise ValueError(f"{self.time_bins} time bins do not divide into patch height {self.patch_h}")
        if self.n_mels % self.patch_w:
            raise ValueError(f"{self.n_mels} mels do not divide into patch width {self.patch_w}")
        if self.embed_dim % self.num_heads:
            raise ValueError(f"{self.embed_dim} is not divisible by {self.num_heads} heads")

    @property
    def patch_size(self) -> tuple[int, int]:
        return (self.patch_h, self.patch_w)

    @property
    def f_max(self) -> float:
        return self.sample_rate / 2

    @property
    def hop_length_ms(self) -> float:
        """Kaldi frame shift in milliseconds."""
        return (self.clip_length * 1000) / self.time_bins

    @property
    def frame_length_ms(self) -> float:
        """Kaldi frame length in milliseconds. Audio-JEPA uses 2.5 hops."""
        return 2.5 * self.hop_length_ms

    @property
    def head_dim(self) -> int:
        return self.embed_dim // self.num_heads

    @property
    def mlp_hidden(self) -> int:
        return int(self.embed_dim * self.mlp_ratio)

    @property
    def grid_time(self) -> int:
        """Patch rows. The spectrogram height is time."""
        return self.time_bins // self.patch_h

    @property
    def grid_freq(self) -> int:
        """Patch columns. The spectrogram width is mel."""
        return self.n_mels // self.patch_w

    @property
    def num_patches(self) -> int:
        return self.grid_time * self.grid_freq

    @property
    def clip_samples(self) -> int:
        return int(self.clip_length * self.sample_rate)


In [ ]:
#| export
def pretrained_config() -> AudioJEPAConfig:
    """Config of `ltuncay/Audio-JEPA`."""
    return AudioJEPAConfig()


In [ ]:
#| export
def tiny_config() -> AudioJEPAConfig:
    """A few small blocks of the same wiring, for graph checks and parity tests."""
    return AudioJEPAConfig(
        time_bins=16,
        n_mels=8,
        patch_h=4,
        patch_w=4,
        embed_dim=32,
        depth=2,
        num_heads=4,
        mlp_ratio=2.0,
        clip_length=1.0,
    )


The published encoder is 16 time patches by 8 frequency patches.


In [ ]:
assert pretrained_config().num_patches == 128
assert pretrained_config().grid_time == 16
assert pretrained_config().grid_freq == 8
assert pretrained_config().hop_length_ms == 10_000 / 256
assert tiny_config().num_patches == 8
